# 2. Multi-Mission Search — All Products

Searches every Sentinel-1, Sentinel-2, Landsat, NISAR, MODIS, and
Sentinel-6 product variant over an AOI and time range, pulling each from
wherever it actually lives — Google Earth Engine for what's in its
catalog, and each product's real archive (ASF DAAC, LP DAAC, PO.DAAC) for
what isn't — and merges everything into one table you can filter and
export. Every row carries the fields most useful for filtering: nominal
spatial resolution (`resolution_m`), processing level (`level`), AOI
coverage %, cloud cover (optical), polarization/mode (SAR), platform,
tile/path-row, file size where the source reports it, and a download link.

| Product | Source | Resolution |
|---|---|---|
| Sentinel-2 L2A (Surface Reflectance) | Earth Engine | 10 m |
| Sentinel-2 L1C (Top-of-Atmosphere) | Earth Engine | 10 m |
| Sentinel-2 HLS S30 (Harmonized) | LP DAAC Cloud | 30 m |
| Sentinel-1 GRD High-Res (Dual-pol) | ASF DAAC | 10 m |
| Sentinel-1 GRD High-Res (Single-pol) | ASF DAAC | 10 m |
| Sentinel-1 GRD Medium-Res (Dual-pol) | ASF DAAC | 40 m |
| Sentinel-1 SLC | ASF DAAC | ~5×20 m |
| Sentinel-1 OCN (ocean) | ASF DAAC | varies (derived product) |
| NISAR L2 GCOV (Geocoded Covariance) | ASF DAAC | ~20 m (provisional) |
| NISAR L1 RSLC (Range-Doppler Complex) | ASF DAAC | ~5 m (provisional, ~25 GB/file) |
| NISAR L3 Soil Moisture (SME2) | ASF DAAC | ~1 km (provisional) |
| Landsat 8 C2 L2 (Surface Reflectance) | Earth Engine | 30 m |
| Landsat 9 C2 L2 (Surface Reflectance) | Earth Engine | 30 m |
| Landsat HLS L30 (Harmonized) | LP DAAC Cloud | 30 m |
| MODIS Terra/Aqua Vegetation (NDVI/EVI) | Earth Engine | 250 m |
| MODIS Terra/Aqua Snow Cover | Earth Engine | 500 m |
| Sentinel-6 Low-Res OST (ocean altimetry) | PO.DAAC | along-track (ocean-only) |

**Requires** an AOI already exported by `1_AOI_Selection.ipynb` (reads its
`.geojson` file — run that notebook first if you haven't).

📖 Stuck, or wondering why a sensor returned 0 results? See
`docs/pdf/02_Search_Guide.pdf`.</cell id="20f8e262">

## Setup — authenticate and initialize Earth Engine

In [ ]:
import ee

EE_PROJECT = "rosy-precinct-498822-e1"  # <-- your GEE-enabled Cloud project

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    # auth_mode="localhost" opens your browser and completes automatically via
    # a local redirect — no authorization code to copy/paste.
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")

## Display settings

By default pandas truncates long tables (limited rows, `...` in wide cells).
This turns that off for the whole notebook so every result table below shows
in full.

In [2]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

## Load your exported AOI

`AOI_NAME` / `OUTPUT_DIR` must match what you used in `1_AOI_Selection.ipynb`.

In [3]:
from pathlib import Path

from aoi_export import load_geometry

AOI_NAME = "my_aoi"
OUTPUT_DIR = "output"

geojson_path = Path(OUTPUT_DIR) / f"{AOI_NAME}.geojson"
if not geojson_path.exists():
    raise FileNotFoundError(
        f"{geojson_path} not found — run 1_AOI_Selection.ipynb first to draw and export an AOI."
    )

geometry = load_geometry(geojson_path)
print("AOI area (km^2):", geometry.area(1).divide(1e6).getInfo())

AOI area (km^2): 438.12151609980384


## (Optional) Preview the AOI on a map

In [4]:
import geemap

preview_map = geemap.Map()
preview_map.add_basemap("Esri.WorldImagery")
preview_map.add_geojson(str(geojson_path), layer_name=AOI_NAME)
preview_map.centerObject(geometry, zoom=10)
preview_map

Map(center=[33.44661989593586, -88.73462950000142], controls=(WidgetControl(options=['position', 'transparent_…

## Search configuration

One shared date range for every sensor below.

In [5]:
START_DATE = "2026-06-01"
END_DATE = "2026-08-01"

## Shared search helpers

Two reusable search functions, used by every sensor section below:

- `search_ee_products(products, aoi, start, end)` — for products in Earth
  Engine's catalog. `products` is `{label: cfg}` where `cfg` has:
  `collection_id`, `resolution_m` (nominal ground sample distance in
  meters), `level` (processing level, e.g. "L2A"), and `properties`
  (`{output_column: earth_engine_property_name}` — each sensor's raw
  metadata field names differ, so this maps them to shared column names
  like `cloud_pct`, `platform`).
- `search_cmr_products(products, aoi, start, end)` — for products queried
  directly from NASA's public CMR API (their real archive, e.g. ASF DAAC,
  LP DAAC). `products` is `{label: cfg}` where `cfg` has: `provider`,
  `short_names`, `id_regex` (parses mode/polarization out of the granule
  ID, or `None`), `resolution_m`, `level`.

Every row also gets `aoi_cov_pct` (% of the AOI the scene covers) and,
where the source reports it, `cloud_pct` and `size_mb` — the fields most
useful for filtering later.

Each scene's real footprint geometry is **not** stored as a table column
(a raw coordinate list would wreck every printed table) — it's saved in a
separate `FOOTPRINTS` dict keyed by `(product, id)`, used only by the map
section at the end. Every other column stays exactly as before.

**To add a new sensor later**: write its own `..._PRODUCTS` dict, call
whichever of these two functions fits its source, and add the resulting
DataFrame to the `pd.concat([...])` list in the "Combine sensors" section
at the bottom — no changes needed here.

In [6]:
import re

import pandas as pd
import requests

CMR_GRANULES_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"

# Scene footprints, keyed by (product, id) -> GeoJSON geometry dict.
# Kept out of the DataFrames so printed tables stay readable; used only
# by the map section at the end of this notebook.
FOOTPRINTS = {}


def _aoi_coverage_pct(image_geom, aoi):
    aoi_area = aoi.area(1)
    inter_area = image_geom.intersection(aoi, 1).area(1)
    return inter_area.divide(aoi_area).multiply(100)


def _sensor_from_product(label):
    return label.split()[0]  # "Sentinel-1 ..." / "Sentinel-2 ..." / "Landsat ..." / "NISAR" -> first token


def _tidy(df):
    if df.empty:
        return df
    front = ["product", "sensor", "id", "date", "level", "resolution_m", "aoi_cov_pct", "cloud_pct"]
    front = [c for c in front if c in df.columns]
    df = df[front + [c for c in df.columns if c not in front]]
    return df.sort_values(["product", "date"]).reset_index(drop=True)


def search_ee_products(products, aoi, start_date, end_date):
    """products: {label: {'collection_id', 'resolution_m', 'level', 'properties'}} -> DataFrame.

    'properties' maps output column name -> Earth Engine image property name,
    e.g. {"cloud_pct": "CLOUDY_PIXEL_PERCENTAGE", "platform": "SPACECRAFT_NAME"}.
    Each row's real footprint is saved into FOOTPRINTS, not the DataFrame.
    """
    rows = []
    for label, cfg in products.items():
        collection = ee.ImageCollection(cfg["collection_id"]).filterBounds(aoi).filterDate(start_date, end_date)
        prop_map = cfg.get("properties", {})

        def add_props(image, prop_map=prop_map):
            props = {
                "id": image.get("system:index"),
                "date": image.date().format("YYYY-MM-dd HH:mm"),
                "aoi_cov_pct": _aoi_coverage_pct(image.geometry(), aoi),
            }
            for out_col, ee_prop in prop_map.items():
                props[out_col] = image.get(ee_prop)
            return ee.Feature(image.geometry(), props)

        fc = ee.FeatureCollection(collection.map(add_props))
        info = fc.getInfo()
        print(f"{label}: {len(info['features'])} scene(s)")
        for feature in info["features"]:
            row = feature["properties"]
            row["product"] = label
            row["sensor"] = _sensor_from_product(label)
            row["resolution_m"] = cfg.get("resolution_m")
            row["level"] = cfg.get("level")
            FOOTPRINTS[(label, row["id"])] = feature["geometry"]
            rows.append(row)

    return _tidy(pd.DataFrame(rows))


def _cmr_bbox(aoi):
    coords = aoi.bounds(1).coordinates().getInfo()[0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return f"{min(lons)},{min(lats)},{max(lons)},{max(lats)}"


def _granule_footprint(granule):
    """GeoJSON Polygon from a CMR granule's 'polygons' field, or None if absent."""
    polygons = granule.get("polygons")
    if not polygons:
        return None
    # CMR polygon strings are "lat lon lat lon ..."; swap to (lon, lat) for GeoJSON/ee.Geometry.
    ring = polygons[0][0].split()
    points = [[float(ring[i + 1]), float(ring[i])] for i in range(0, len(ring), 2)]
    return {"type": "Polygon", "coordinates": [points]}


def _granule_link(granule):
    data_links = [l["href"] for l in granule.get("links", []) if "/data#" in l.get("rel", "")]
    zip_links = [href for href in data_links if href.lower().endswith(".zip")]
    if zip_links:
        return zip_links[0]
    return data_links[0] if data_links else None


def search_cmr_products(products, aoi, start_date, end_date, page_size=200):
    """products: {label: {'provider', 'short_names', 'id_regex', 'resolution_m', 'level'}} -> DataFrame.

    'id_regex' can have any named groups (e.g. 'instrument_mode', 'polarization',
    'orbit_pass') — whatever it captures becomes a column automatically. Each
    row's real footprint (from CMR) is saved into FOOTPRINTS, not the DataFrame.
    """
    rows = []
    for label, cfg in products.items():
        params = [("short_name", s) for s in cfg["short_names"]]
        params += [
            ("provider", cfg["provider"]),
            ("bounding_box", _cmr_bbox(aoi)),
            ("temporal", f"{start_date}T00:00:00Z,{end_date}T23:59:59Z"),
            ("page_size", str(page_size)),
        ]
        resp = requests.get(CMR_GRANULES_URL, params=params, timeout=60)
        resp.raise_for_status()
        entries = resp.json()["feed"]["entry"]
        print(f"{label}: {len(entries)} scene(s)")

        id_regex = re.compile(cfg["id_regex"]) if cfg.get("id_regex") else None
        for g in entries:
            granule_id = g.get("producer_granule_id") or g.get("title", "")
            match = id_regex.match(granule_id) if id_regex else None
            footprint = _granule_footprint(g)
            cov_pct = (
                _aoi_coverage_pct(ee.Geometry(footprint), aoi).getInfo() if footprint else None
            )
            if footprint:
                FOOTPRINTS[(label, granule_id)] = footprint
            row = {
                "product": label,
                "sensor": _sensor_from_product(label),
                "id": granule_id,
                "date": g.get("time_start"),
                "resolution_m": cfg.get("resolution_m"),
                "level": cfg.get("level"),
                "aoi_cov_pct": cov_pct,
                "cloud_pct": float(g["cloud_cover"]) if g.get("cloud_cover") not in (None, "") else None,
                "size_mb": float(g["granule_size"]) if g.get("granule_size") not in (None, "") else None,
                "link": _granule_link(g),
            }
            if match:
                row.update(match.groupdict())
            rows.append(row)

    return _tidy(pd.DataFrame(rows))

## Sentinel-2 — all products together

L2A and L1C come from Earth Engine; HLS S30 isn't in Earth Engine's
catalog, so it's queried directly from LP DAAC Cloud via CMR. Both go into
one `sentinel2_df`.

In [7]:
S2_EE_PRODUCTS = {
    "Sentinel-2 L2A (Surface Reflectance)": {
        "collection_id": "COPERNICUS/S2_SR_HARMONIZED",
        "resolution_m": 10,
        "level": "L2A",
        "properties": {
            "cloud_pct": "CLOUDY_PIXEL_PERCENTAGE",
            "tile": "MGRS_TILE",
            "platform": "SPACECRAFT_NAME",
        },
    },
    "Sentinel-2 L1C (Top-of-Atmosphere)": {
        "collection_id": "COPERNICUS/S2_HARMONIZED",
        "resolution_m": 10,
        "level": "L1C",
        "properties": {
            "cloud_pct": "CLOUDY_PIXEL_PERCENTAGE",
            "tile": "MGRS_TILE",
            "platform": "SPACECRAFT_NAME",
        },
    },
}

S2_CMR_PRODUCTS = {
    "Sentinel-2 HLS S30": {
        "provider": "LPCLOUD",
        "short_names": ["HLSS30"],
        "id_regex": None,
        "resolution_m": 30,
        "level": "HLS S30",
    },
}

sentinel2_df = pd.concat(
    [
        search_ee_products(S2_EE_PRODUCTS, geometry, START_DATE, END_DATE),
        search_cmr_products(S2_CMR_PRODUCTS, geometry, START_DATE, END_DATE),
    ],
    ignore_index=True,
)
sentinel2_df

Sentinel-2 L2A (Surface Reflectance): 60 scene(s)
Sentinel-2 L1C (Top-of-Atmosphere): 60 scene(s)
Sentinel-2 HLS S30: 50 scene(s)


,product,sensor,id,date,level,resolution_m,aoi_cov_pct,cloud_pct,platform,tile,size_mb,link
0,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260602T163701_20260602T163932_T16SCB,2026-06-02 16:44,L1C,10,38.683796,19.214399,Sentinel-2A,16SCB,NaN,NaN
1,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260602T163701_20260602T163932_T16SCC,2026-06-02 16:44,L1C,10,100.000000,88.774523,Sentinel-2A,16SCC,NaN,NaN
2,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260603T163841_20260603T165330_T16SCB,2026-06-03 16:54,L1C,10,14.307021,72.234374,Sentinel-2C,16SCB,NaN,NaN
3,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260603T163841_20260603T165330_T16SCC,2026-06-03 16:54,L1C,10,44.038025,17.143584,Sentinel-2C,16SCC,NaN,NaN
4,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260605T162829_20260605T163621_T16SCB,2026-06-05 16:44,L1C,10,38.683796,15.881334,Sentinel-2B,16SCB,NaN,NaN
5,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260605T162829_20260605T163621_T16SCC,2026-06-05 16:44,L1C,10,100.000000,58.051838,Sentinel-2B,16SCC,NaN,NaN
6,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260608T163859_20260608T164740_T16SCB,2026-06-08 16:54,L1C,10,15.182945,89.788520,Sentinel-2B,16SCB,NaN,NaN
7,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260608T163859_20260608T164740_T16SCC,2026-06-08 16:54,L1C,10,46.366464,82.351505,Sentinel-2B,16SCC,NaN,NaN
8,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260610T162831_20260610T163352_T16SCB,2026-06-10 16:44,L1C,10,38.683796,0.426498,Sentinel-2C,16SCB,NaN,NaN
9,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260610T162831_20260610T163352_T16SCC,2026-06-10 16:44,L1C,10,100.000000,21.682732,Sentinel-2C,16SCC,NaN,NaN


## Sentinel-1 — all products together

None of these are in Earth Engine's catalog (it only has generic GRD),
so every variant is queried directly from ASF DAAC via CMR. Single-pol,
medium-res, and OCN commonly return 0 scenes over land AOIs — that's a
real acquisition-pattern fact (e.g. OCN is only produced over ocean), not
a search bug.

`resolution_m` is the nominal figure ESA quotes per product type (pixel
spacing for GRD, finer dimension for SLC's anisotropic ~5×20 m IW
resolution). OCN is a derived geophysical product (wind/wave/current),
not a fixed image grid, so it has no single resolution value.

In [8]:
_S1_ID_RE = r"^S1[A-Z]_(?P<instrument_mode>IW|EW|SM|WV)_{ptype}(?P<polarization>SH|SV|DH|DV)_"

S1_CMR_PRODUCTS = {
    "Sentinel-1 GRD High-Res (Dual-pol)": {
        "provider": "ASF",
        "short_names": ["SENTINEL-1A_DP_GRD_HIGH", "SENTINEL-1B_DP_GRD_HIGH", "SENTINEL-1C_DP_GRD_HIGH"],
        "id_regex": _S1_ID_RE.format(ptype="GRDH_1S"),
        "resolution_m": 10,
        "level": "GRDH",
    },
    "Sentinel-1 GRD High-Res (Single-pol)": {
        "provider": "ASF",
        "short_names": ["SENTINEL-1A_SP_GRD_HIGH", "SENTINEL-1B_SP_GRD_HIGH", "SENTINEL-1C_SP_GRD_HIGH"],
        "id_regex": _S1_ID_RE.format(ptype="GRDH_1S"),
        "resolution_m": 10,
        "level": "GRDH",
    },
    "Sentinel-1 GRD Medium-Res (Dual-pol)": {
        "provider": "ASF",
        "short_names": ["SENTINEL-1A_DP_GRD_MEDIUM", "SENTINEL-1B_DP_GRD_MEDIUM", "SENTINEL-1C_DP_GRD_MEDIUM"],
        "id_regex": _S1_ID_RE.format(ptype="GRDM_1S"),
        "resolution_m": 40,
        "level": "GRDM",
    },
    "Sentinel-1 SLC": {
        "provider": "ASF",
        "short_names": ["SENTINEL-1A_SLC", "SENTINEL-1B_SLC", "SENTINEL-1C_SLC"],
        "id_regex": _S1_ID_RE.format(ptype="SLC__1S"),
        "resolution_m": 5,
        "level": "SLC",
    },
    "Sentinel-1 OCN": {
        "provider": "ASF",
        "short_names": ["SENTINEL-1A_OCN", "SENTINEL-1B_OCN", "SENTINEL-1C_OCN"],
        "id_regex": _S1_ID_RE.format(ptype="OCN__2S"),
        "resolution_m": None,
        "level": "OCN",
    },
}

sentinel1_df = search_cmr_products(S1_CMR_PRODUCTS, geometry, START_DATE, END_DATE)
sentinel1_df

Sentinel-1 GRD High-Res (Dual-pol): 2 scene(s)
Sentinel-1 GRD High-Res (Single-pol): 0 scene(s)
Sentinel-1 GRD Medium-Res (Dual-pol): 0 scene(s)
Sentinel-1 SLC: 2 scene(s)
Sentinel-1 OCN: 0 scene(s)


,product,sensor,id,date,level,resolution_m,aoi_cov_pct,cloud_pct,size_mb,link,instrument_mode,polarization
0,Sentinel-1 GRD High-Res (Dual-pol),Sentinel-1,S1A_IW_GRDH_1SDV_20260610T235457_20260610T235522_064914_082E20_CF54,2026-06-10T23:54:57.895Z,GRDH,10,100,None,969.349243,https://datapool.asf.alaska.edu/GRD_HD/SA/S1A_IW_GRDH_1SDV_20260610T235457_20260610T235522_064914_082E20_CF54.zip,IW,DV
1,Sentinel-1 GRD High-Res (Dual-pol),Sentinel-1,S1A_IW_GRDH_1SDV_20260622T235457_20260622T235522_065089_083447_3A91,2026-06-22T23:54:57.254Z,GRDH,10,100,None,972.420809,https://datapool.asf.alaska.edu/GRD_HD/SA/S1A_IW_GRDH_1SDV_20260622T235457_20260622T235522_065089_083447_3A91.zip,IW,DV
2,Sentinel-1 SLC,Sentinel-1,S1A_IW_SLC__1SDV_20260610T235457_20260610T235523_064914_082E20_6CEC,2026-06-10T23:54:57.022Z,SLC,5,100,None,4538.479953,https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_SLC__1SDV_20260610T235457_20260610T235523_064914_082E20_6CEC.zip,IW,DV
3,Sentinel-1 SLC,Sentinel-1,S1A_IW_SLC__1SDV_20260622T235456_20260622T235523_065089_083447_C8C0,2026-06-22T23:54:56.377Z,SLC,5,100,None,4558.741018,https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_SLC__1SDV_20260622T235456_20260622T235523_065089_083447_C8C0.zip,IW,DV


## NISAR — all products together

NISAR (NASA-ISRO SAR) is a newer L/S-band radar mission, not in Earth
Engine's catalog — queried directly from ASF DAAC via CMR, same mechanism
as Sentinel-1. Data is still in its **provisional** early-mission tier
(the mission is in its commissioning/early-science phase), so resolution
figures below are approximate and may be refined as the mission matures.

- **GCOV** (Geocoded Covariance) — the standard ready-to-use backscatter
  product, closest analog to Sentinel-1 GRD.
- **RSLC** (Range-Doppler Single Look Complex) — raw-ish complex data,
  closest analog to Sentinel-1 SLC. **Files are enormous — around 25 GB
  each**, noticeably bigger than even Sentinel-1 SLC.
- **L3 Soil Moisture (SME2)** — a derived geophysical product, directly
  relevant for agricultural monitoring. Much smaller (~100–150 MB).

`orbit_pass` (A/D = Ascending/Descending) is parsed from the granule ID,
the one field NISAR's naming reliably exposes without guessing at the
mission's more complex dual-frequency/dual-pol encoding.

In [9]:
_NISAR_ID_RE = r"^NISAR_L\d_PR_[A-Z0-9]+_\d+_\d+_(?P<orbit_pass>[AD])_"

NISAR_CMR_PRODUCTS = {
    "NISAR L2 GCOV (Geocoded Covariance)": {
        "provider": "ASF",
        "short_names": ["NISAR_L2_GCOV_PROVISIONAL_V1"],
        "id_regex": _NISAR_ID_RE,
        "resolution_m": 20,
        "level": "L2 GCOV",
    },
    "NISAR L1 RSLC (Range-Doppler Complex)": {
        "provider": "ASF",
        "short_names": ["NISAR_L1_RSLC_PROVISIONAL_V1"],
        "id_regex": _NISAR_ID_RE,
        "resolution_m": 5,
        "level": "L1 RSLC",
    },
    "NISAR L3 Soil Moisture": {
        "provider": "ASF",
        "short_names": ["NISAR_L3_SME2_PROVISIONAL_V1"],
        "id_regex": _NISAR_ID_RE,
        "resolution_m": 1000,
        "level": "L3 SME2",
    },
}

nisar_df = search_cmr_products(NISAR_CMR_PRODUCTS, geometry, START_DATE, END_DATE)
nisar_df

NISAR L2 GCOV (Geocoded Covariance): 23 scene(s)
NISAR L1 RSLC (Range-Doppler Complex): 23 scene(s)
NISAR L3 Soil Moisture: 18 scene(s)


,product,sensor,id,date,level,resolution_m,aoi_cov_pct,cloud_pct,size_mb,link,orbit_pass
0,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_021_156_D_072_4005_DHDH_A_20260601T004440_20260601T004515_P05023_N_F_J_001,2026-06-01T00:44:40.000Z,L1 RSLC,5,100.000000,None,25184.422103,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_021_156_D_072_4005_DHDH_A_20260601T004440_20260601T004515_P05023_N_F_J_001/NISAR_L1_PR_RSLC_021_156_D_072_4005_DHDH_A_20260601T004440_20260601T004515_P05023_N_F_J_001.h5,D
1,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_022_004_A_018_4005_DHDH_A_20260602T111121_20260602T111156_P05023_N_F_J_001,2026-06-02T11:11:21.000Z,L1 RSLC,5,100.000000,None,25224.238111,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_022_004_A_018_4005_DHDH_A_20260602T111121_20260602T111156_P05023_N_F_J_001/NISAR_L1_PR_RSLC_022_004_A_018_4005_DHDH_A_20260602T111121_20260602T111156_P05023_N_F_J_001.h5,A
2,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_022_076_A_018_4005_DHDH_A_20260607T110302_20260607T110336_P05023_N_F_J_001,2026-06-07T11:03:02.000Z,L1 RSLC,5,0.700452,None,24507.844208,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_022_076_A_018_4005_DHDH_A_20260607T110302_20260607T110336_P05023_N_F_J_001/NISAR_L1_PR_RSLC_022_076_A_018_4005_DHDH_A_20260607T110302_20260607T110336_P05023_N_F_J_001.h5,A
3,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_022_156_D_072_4005_DHDH_A_20260613T004439_20260613T004514_P05023_N_F_J_001,2026-06-13T00:44:39.000Z,L1 RSLC,5,100.000000,None,25184.499689,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_022_156_D_072_4005_DHDH_A_20260613T004439_20260613T004514_P05023_N_F_J_001/NISAR_L1_PR_RSLC_022_156_D_072_4005_DHDH_A_20260613T004439_20260613T004514_P05023_N_F_J_001.h5,D
4,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_023_004_A_018_4005_DHDH_A_20260614T111120_20260614T111155_P05023_N_F_J_001,2026-06-14T11:11:20.000Z,L1 RSLC,5,100.000000,None,25224.295504,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_023_004_A_018_4005_DHDH_A_20260614T111120_20260614T111155_P05023_N_F_J_001/NISAR_L1_PR_RSLC_023_004_A_018_4005_DHDH_A_20260614T111120_20260614T111155_P05023_N_F_J_001.h5,A
5,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_023_076_A_018_4005_DHDH_A_20260619T110301_20260619T110336_P05023_N_F_J_001,2026-06-19T11:03:01.000Z,L1 RSLC,5,5.379086,None,25220.642879,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_023_076_A_018_4005_DHDH_A_20260619T110301_20260619T110336_P05023_N_F_J_001/NISAR_L1_PR_RSLC_023_076_A_018_4005_DHDH_A_20260619T110301_20260619T110336_P05023_N_F_J_001.h5,A
6,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_023_076_A_018_4005_DHDH_A_20260619T110301_20260619T110336_P05023_N_F_J_002,2026-06-19T11:03:01.000Z,L1 RSLC,5,5.379086,None,25220.642947,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_023_076_A_018_4005_DHDH_A_20260619T110301_20260619T110336_P05023_N_F_J_002/NISAR_L1_PR_RSLC_023_076_A_018_4005_DHDH_A_20260619T110301_20260619T110336_P05023_N_F_J_002.h5,A
7,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_023_076_A_019_4005_DHDH_A_20260619T110335_20260619T110409_P05023_N_F_J_001,2026-06-19T11:03:35.000Z,L1 RSLC,5,22.459990,None,24520.374425,https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_023_076_A_019_4005_DHDH_A_20260619T110335_20260619T110409_P05023_N_F_J_001/NISAR_L1_PR_RSLC_023_076_A_019_4005_DHDH_A_20260619T110335_20260619T110409_P05023_N_F_J_001.h5,A
8,NISAR L1 RSLC (Range-Doppler Complex),NISAR,NISAR_L1_PR_RSLC_023_156_D_071_4005_DHDH_A_20260625T004405_20260625T004440_P05023_N_F_J_001,2026-06-25T00:44:05.000Z,L1 RSLC,5,2

## Landsat — all products together

Landsat 8/9 Collection 2 Level-2 (Surface Reflectance) are both in Earth
Engine's catalog directly. HLS L30 (Landsat harmonized to the same 30 m
grid as Sentinel-2's HLS S30, useful for combined S2+Landsat time series)
isn't in Earth Engine, so it's queried from LP DAAC Cloud via CMR, same as
Sentinel-2's HLS S30 above.

In [10]:
LANDSAT_EE_PRODUCTS = {
    "Landsat 8 C2 L2 (Surface Reflectance)": {
        "collection_id": "LANDSAT/LC08/C02/T1_L2",
        "resolution_m": 30,
        "level": "L2 SR",
        "properties": {
            "cloud_pct": "CLOUD_COVER",
            "path": "WRS_PATH",
            "row": "WRS_ROW",
            "platform": "SPACECRAFT_ID",
            "instrument": "SENSOR_ID",
        },
    },
    "Landsat 9 C2 L2 (Surface Reflectance)": {
        "collection_id": "LANDSAT/LC09/C02/T1_L2",
        "resolution_m": 30,
        "level": "L2 SR",
        "properties": {
            "cloud_pct": "CLOUD_COVER",
            "path": "WRS_PATH",
            "row": "WRS_ROW",
            "platform": "SPACECRAFT_ID",
            "instrument": "SENSOR_ID",
        },
    },
}

LANDSAT_CMR_PRODUCTS = {
    "Landsat HLS L30": {
        "provider": "LPCLOUD",
        "short_names": ["HLSL30"],
        "id_regex": None,
        "resolution_m": 30,
        "level": "HLS L30",
    },
}

landsat_df = pd.concat(
    [
        search_ee_products(LANDSAT_EE_PRODUCTS, geometry, START_DATE, END_DATE),
        search_cmr_products(LANDSAT_CMR_PRODUCTS, geometry, START_DATE, END_DATE),
    ],
    ignore_index=True,
)
landsat_df

Landsat 8 C2 L2 (Surface Reflectance): 8 scene(s)
Landsat 9 C2 L2 (Surface Reflectance): 7 scene(s)
Landsat HLS L30: 32 scene(s)


,product,sensor,id,date,level,resolution_m,aoi_cov_pct,cloud_pct,instrument,path,platform,row,size_mb,link
0,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_021037_20260606,2026-06-06 16:24,L2 SR,30,61.691692,44.52,OLI_TIRS,21.0,LANDSAT_8,37.0,NaN,NaN
1,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_022037_20260613,2026-06-13 16:30,L2 SR,30,100.000000,52.65,OLI_TIRS,22.0,LANDSAT_8,37.0,NaN,NaN
2,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_021037_20260622,2026-06-22 16:24,L2 SR,30,65.346238,78.37,OLI_TIRS,21.0,LANDSAT_8,37.0,NaN,NaN
3,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_022037_20260629,2026-06-29 16:30,L2 SR,30,100.000000,17.97,OLI_TIRS,22.0,LANDSAT_8,37.0,NaN,NaN
4,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_021037_20260708,2026-07-08 16:24,L2 SR,30,63.606452,33.13,OLI_TIRS,21.0,LANDSAT_8,37.0,NaN,NaN
5,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_022037_20260715,2026-07-15 16:30,L2 SR,30,100.000000,65.24,OLI_TIRS,22.0,LANDSAT_8,37.0,NaN,NaN
6,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_021037_20260724,2026-07-24 16:24,L2 SR,30,62.731123,99.99,OLI_TIRS,21.0,LANDSAT_8,37.0,NaN,NaN
7,Landsat 8 C2 L2 (Surface Reflectance),Landsat,LC08_022037_20260731,2026-07-31 16:30,L2 SR,30,100.000000,18.93,OLI_TIRS,22.0,LANDSAT_8,37.0,NaN,NaN
8,Landsat 9 C2 L2 (Surface Reflectance),Landsat,LC09_022037_20260605,2026-06-05 16:30,L2 SR,30,100.000000,14.50,OLI_TIRS,22.0,LANDSAT_9,37.0,NaN,NaN
9,Landsat 9 C2 L2 (Surface Reflectance),Landsat,LC09_021037_20260614,2026-06-14 16:24,L2 SR,30,63.772301,20.36,OLI_TIRS,21.0,LANDSAT_9,37.0,NaN,NaN


## MODIS — Vegetation & Snow

Daily-to-16-day global vegetation index and snow cover products — much
more frequent revisit than Sentinel-2/Landsat, useful for filling gaps
between their passes. All in Earth Engine's catalog. Terra (`MOD`) and
Aqua (`MYD`) are two separate satellites carrying identical MODIS
instruments, offset in time from each other.

**Note**: these are global daily/composite mosaics, not swath-limited
scenes — every result will show `aoi_cov_pct` ≈ 100%, since the "scene"
already covers the whole globe. That filter isn't meaningful for these
products specifically; use the date/product columns instead.

In [11]:
MODIS_EE_PRODUCTS = {
    "MODIS Terra Vegetation (NDVI/EVI)": {
        "collection_id": "MODIS/061/MOD13Q1",
        "resolution_m": 250,
        "level": "L3 VI 16-day",
        "properties": {},
    },
    "MODIS Aqua Vegetation (NDVI/EVI)": {
        "collection_id": "MODIS/061/MYD13Q1",
        "resolution_m": 250,
        "level": "L3 VI 16-day",
        "properties": {},
    },
    "MODIS Terra Snow Cover": {
        "collection_id": "MODIS/061/MOD10A1",
        "resolution_m": 500,
        "level": "L3 Snow Daily",
        "properties": {},
    },
    "MODIS Aqua Snow Cover": {
        "collection_id": "MODIS/061/MYD10A1",
        "resolution_m": 500,
        "level": "L3 Snow Daily",
        "properties": {},
    },
}

modis_df = search_ee_products(MODIS_EE_PRODUCTS, geometry, START_DATE, END_DATE)
modis_df

MODIS Terra Vegetation (NDVI/EVI): 4 scene(s)
MODIS Aqua Vegetation (NDVI/EVI): 4 scene(s)
MODIS Terra Snow Cover: 61 scene(s)
MODIS Aqua Snow Cover: 61 scene(s)


,product,sensor,id,date,level,resolution_m,aoi_cov_pct
0,MODIS Aqua Snow Cover,MODIS,2026_06_01,2026-06-01 00:00,L3 Snow Daily,500,100
1,MODIS Aqua Snow Cover,MODIS,2026_06_02,2026-06-02 00:00,L3 Snow Daily,500,100
2,MODIS Aqua Snow Cover,MODIS,2026_06_03,2026-06-03 00:00,L3 Snow Daily,500,100
3,MODIS Aqua Snow Cover,MODIS,2026_06_04,2026-06-04 00:00,L3 Snow Daily,500,100
4,MODIS Aqua Snow Cover,MODIS,2026_06_05,2026-06-05 00:00,L3 Snow Daily,500,100
5,MODIS Aqua Snow Cover,MODIS,2026_06_06,2026-06-06 00:00,L3 Snow Daily,500,100
6,MODIS Aqua Snow Cover,MODIS,2026_06_07,2026-06-07 00:00,L3 Snow Daily,500,100
7,MODIS Aqua Snow Cover,MODIS,2026_06_08,2026-06-08 00:00,L3 Snow Daily,500,100
8,MODIS Aqua Snow Cover,MODIS,2026_06_09,2026-06-09 00:00,L3 Snow Daily,500,100
9,MODIS Aqua Snow Cover,MODIS,2026_06_10,2026-06-10 00:00,L3 Snow Daily,500,100


## Sentinel-6 — Ocean Surface Altimetry

Radar altimeter mission (Jason-CS/Sentinel-6 Michael Freilich), not in
Earth Engine's catalog — queried directly from PO.DAAC via CMR, same
mechanism as Sentinel-1/NISAR. **This is an ocean-only mission** — it
measures sea surface height along the satellite's ground track, so it
will correctly return 0 scenes for any land-locked AOI (that's real
mission coverage, not a search bug). Orbit inclination is 66°, so it also
returns nothing above/below ±66° latitude. Footprints here are wide
along-track polygons rather than compact scene boundaries — a different
shape than the other products, but handled by the same footprint logic.

In [12]:
S6_CMR_PRODUCTS = {
    "Sentinel-6 Low-Res OST (NTC)": {
        "provider": "POCLOUD",
        "short_names": ["JASON_CS_S6A_L2_ALT_LR_STD_OST_NTC_F08"],
        "id_regex": None,
        "resolution_m": None,
        "level": "L2 ALT LR OST NTC",
    },
}

sentinel6_df = search_cmr_products(S6_CMR_PRODUCTS, geometry, START_DATE, END_DATE)
sentinel6_df

Sentinel-6 Low-Res OST (NTC): 0 scene(s)


""


## Combine sensors

Add a future sensor by giving it its own `..._df` above, then listing it
here too — nothing else in this notebook needs to change.

In [13]:
# Add future sensors here, e.g.: results_df = pd.concat([..., new_df], ...)
results_df = pd.concat(
    [sentinel2_df, sentinel1_df, nisar_df, landsat_df, modis_df, sentinel6_df],
    ignore_index=True,
)
results_df

,product,sensor,id,date,level,resolution_m,aoi_cov_pct,cloud_pct,platform,tile,size_mb,link,instrument_mode,polarization,orbit_pass,instrument,path,row
0,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260602T163701_20260602T163932_T16SCB,2026-06-02 16:44,L1C,10,38.683796,19.214399,Sentinel-2A,16SCB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260602T163701_20260602T163932_T16SCC,2026-06-02 16:44,L1C,10,100.000000,88.774523,Sentinel-2A,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260603T163841_20260603T165330_T16SCB,2026-06-03 16:54,L1C,10,14.307021,72.234374,Sentinel-2C,16SCB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260603T163841_20260603T165330_T16SCC,2026-06-03 16:54,L1C,10,44.038025,17.143584,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260605T162829_20260605T163621_T16SCB,2026-06-05 16:44,L1C,10,38.683796,15.881334,Sentinel-2B,16SCB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260605T162829_20260605T163621_T16SCC,2026-06-05 16:44,L1C,10,100.000000,58.051838,Sentinel-2B,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260608T163859_20260608T164740_T16SCB,2026-06-08 16:54,L1C,10,15.182945,89.788520,Sentinel-2B,16SCB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260608T163859_20260608T164740_T16SCC,2026-06-08 16:54,L1C,10,46.366464,82.351505,Sentinel-2B,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260610T162831_20260610T163352_T16SCB,2026-06-10 16:44,L1C,10,38.683796,0.426498,Sentinel-2C,16SCB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260610T162831_20260610T163352_T16SCC,2026-06-10 16:44,L1C,10,100.000000,21.682732,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Filter and export the scene list

Keep only scenes that meaningfully cover the AOI and (for optical products)
aren't too cloudy, then save the filtered list to CSV. SAR rows have no
`cloud_pct`, so the cloud filter is skipped for them automatically.

In [14]:
MIN_AOI_COVERAGE_PCT = 50
MAX_CLOUD_PCT = 30

if results_df.empty:
    filtered_df = results_df.copy()
else:
    if "cloud_pct" in results_df.columns:
        cloud_ok = results_df["cloud_pct"].isna() | (results_df["cloud_pct"] <= MAX_CLOUD_PCT)
    else:
        cloud_ok = pd.Series(True, index=results_df.index)

    filtered_df = results_df[
        (results_df["aoi_cov_pct"] >= MIN_AOI_COVERAGE_PCT) & cloud_ok
    ].reset_index(drop=True)

out_csv = Path(OUTPUT_DIR) / f"{AOI_NAME}_scene_search.csv"
filtered_df.to_csv(out_csv, index=False)
print(f"Kept {len(filtered_df)} of {len(results_df)} scenes -> {out_csv.resolve()}")
filtered_df

Kept 180 of 415 scenes -> C:\Users\Say70\OneDrive - Mississippi State University\Desktop\Satellite Data Search\output\my_aoi_scene_search.csv


,product,sensor,id,date,level,resolution_m,aoi_cov_pct,cloud_pct,platform,tile,size_mb,link,instrument_mode,polarization,orbit_pass,instrument,path,row
0,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260610T162831_20260610T163352_T16SCC,2026-06-10 16:44,L1C,10,100.000000,21.682732,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260630T162831_20260630T164408_T16SCC,2026-06-30 16:44,L1C,10,100.000000,19.854972,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260720T162831_20260720T164335_T16SCC,2026-07-20 16:44,L1C,10,100.000000,0.740392,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sentinel-2 L1C (Top-of-Atmosphere),Sentinel-2,20260730T162841_20260730T163429_T16SCC,2026-07-30 16:44,L1C,10,100.000000,18.076383,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sentinel-2 L2A (Surface Reflectance),Sentinel-2,20260630T162831_20260630T164408_T16SCC,2026-06-30 16:44,L2A,10,100.000000,26.040971,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Sentinel-2 L2A (Surface Reflectance),Sentinel-2,20260720T162831_20260720T164335_T16SCC,2026-07-20 16:44,L2A,10,100.000000,2.262594,Sentinel-2C,16SCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Sentinel-2 HLS S30,Sentinel-2,HLS.S30.T16SCC.2026201T162831.v2.0,2026-07-20T16:44:14.711Z,HLS S30,30,100.000000,6.000000,NaN,NaN,NaN,https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSS30.020/HLS.S30.T16SCC.2026201T162831.v2.0/HLS.S30.T16SCC.2026201T162831.v2.0.B07.tif,NaN,NaN,NaN,NaN,NaN,NaN
7,Sentinel-1 GRD High-Res (Dual-pol),Sentinel-1,S1A_IW_GRDH_1SDV_20260610T235457_20260610T235522_064914_082E20_CF54,2026-06-10T23:54:57.895Z,GRDH,10,100.000000,NaN,NaN,NaN,969.349243,https://datapool.asf.alaska.edu/GRD_HD/SA/S1A_IW_GRDH_1SDV_20260610T235457_20260610T235522_064914_082E20_CF54.zip,IW,DV,NaN,NaN,NaN,NaN
8,Sentinel-1 GRD High-Res (Dual-pol),Sentinel-1,S1A_IW_GRDH_1SDV_20260622T235457_20260622T235522_065089_083447_3A91,2026-06-22T23:54:57.254Z,GRDH,10,100.000000,NaN,NaN,NaN,972.420809,https://datapool.asf.alaska.edu/GRD_HD/SA/S1A_IW_GRDH_1SDV_20260622T235457_20260622T235522_065089_083447_3A91.zip,IW,DV,NaN,NaN,NaN,NaN
9,Sentinel-1 SLC,Sentinel-1,S1A_IW_SLC__1SDV_20260610T235457_20260610T235523_064914_082E20_6CEC,2026-06-10T23:54:57.022Z,SLC,5,100.000000,NaN,NaN,NaN,4538.479953,https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_SLC__1SDV_20260610T235457_20260610T235523_064914_082E20_6CEC.zip,IW,DV,NaN,NaN,NaN,NaN


## Visualize filtered scene boundaries on the map

Draws the AOI and every filtered scene's real footprint as colored
outlines (no fill, so overlapping scenes stay readable) on one map — the
AOI in its own color, each sensor in its own color, with a legend. Uses
`ee.Image.paint()` to render boundaries only, looking footprints up from
`FOOTPRINTS` by each row's `(product, id)`, and skips rows with no
footprint on record (a handful of CMR sources occasionally omit it).

Note: MODIS is a global mosaic — its "footprint" covers the whole world,
so at AOI zoom level you won't see a meaningful boundary for it (nothing
wrong, just not a useful visual for that product).

In [ ]:
AOI_COLOR = "#ef4444"  # red
SENSOR_COLORS = {
    "Sentinel-1": "#14b8a6",  # teal
    "Sentinel-2": "#eab308",  # gold
    "Landsat": "#f97316",  # orange
    "NISAR": "#a855f7",  # purple
    "MODIS": "#0ea5e9",  # sky blue
    "Sentinel-6": "#ec4899",  # pink
}


def _outline(fc_or_geom, color, width):
    """Render a geometry/FeatureCollection as a boundary-only (unfilled) image layer."""
    fc = fc_or_geom if isinstance(fc_or_geom, ee.FeatureCollection) else ee.FeatureCollection([ee.Feature(fc_or_geom)])
    return ee.Image().byte().paint(fc, 1, width).visualize(palette=[color])


boundary_map = geemap.Map()
boundary_map.add_basemap("Esri.WorldImagery")

boundary_map.addLayer(_outline(geometry, AOI_COLOR, 3), {}, "AOI")

legend_dict = {"AOI": AOI_COLOR}
for sensor, color in SENSOR_COLORS.items():
    subset = filtered_df[filtered_df["sensor"] == sensor]
    features = []
    for _, row in subset.iterrows():
        footprint = FOOTPRINTS.get((row["product"], row["id"]))
        if footprint:
            features.append(ee.Feature(ee.Geometry(footprint), {"id": row["id"]}))
    if not features:
        continue
    fc = ee.FeatureCollection(features)
    boundary_map.addLayer(_outline(fc, color, 2), {}, f"{sensor} ({len(features)} scenes)")
    legend_dict[f"{sensor} ({len(features)})"] = color

boundary_map.add_legend(title="Boundaries", legend_dict=legend_dict)
boundary_map.centerObject(geometry, zoom=9)
boundary_map

Map(center=[33.44661989593467, -88.73462949999968], controls=(WidgetControl(options=['position', 'transparent_…

: 